# Exercise: RDD Basics

Practice creating RDDs from different sources and applying basic transformations like `map`, `filter`, and `flatMap`.

**Duration:** 45-60 minutes &nbsp;|&nbsp; **Mode:** Individual

## Setup

Create the `SparkContext` used throughout the notebook.

In [1]:
from pyspark import SparkContext

sc = SparkContext("local[*]", "RDDBasics")
sc

<SparkContext master=local[*] appName=RDDBasics>

## Task 1: Create RDDs from Different Sources

In [2]:
# 1. Create RDD from a Python list
numbers = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f"Numbers: {numbers.collect()}")
print(f"Partitions: {numbers.getNumPartitions()}")

Numbers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Partitions: 2


In [3]:
# 2. Create the same list with exactly 4 partitions
numbers_4p = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], numSlices=4)
print(f"Numbers (4 partitions): {numbers_4p.collect()}")
print(f"Partitions: {numbers_4p.getNumPartitions()}")

Numbers (4 partitions): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Partitions: 4


In [4]:
# 3. Create RDD from range(1, 101)
range_rdd = sc.parallelize(range(1, 101))
print(f"Count: {range_rdd.count()}")
print(f"First 10: {range_rdd.take(10)}")
print(f"Last 10: {range_rdd.collect()[-10:]}")

Count: 100
First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last 10: [91, 92, 93, 94, 95, 96, 97, 98, 99, 100]


## Task 2: Apply `map()` Transformation

In [5]:
# Task A: Square each number
squared = numbers.map(lambda x: x ** 2)
print(f"Squared: {squared.collect()}")

Squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]


In [6]:
# Task B: Convert to strings with prefix
prefixed = numbers.map(lambda x: f"num_{x}")
print(f"Prefixed: {prefixed.collect()}")

Prefixed: ['num_1', 'num_2', 'num_3', 'num_4', 'num_5', 'num_6', 'num_7', 'num_8', 'num_9', 'num_10']


## Task 3: Apply `filter()` Transformation

In [7]:
# Task A: Keep only even numbers
# Expected: [2, 4, 6, 8, 10]
evens = numbers.filter(lambda x: x % 2 == 0)
print(f"Evens: {evens.collect()}")

Evens: [2, 4, 6, 8, 10]


In [8]:
# Task B: Keep numbers greater than 5
greater_than_5 = numbers.filter(lambda x: x > 5)
print(f"Greater than 5: {greater_than_5.collect()}")

Greater than 5: [6, 7, 8, 9, 10]


In [9]:
# Task C: Combine - even AND greater than 5
combined = numbers.filter(lambda x: x % 2 == 0 and x > 5)
print(f"Even AND > 5: {combined.collect()}")

Even AND > 5: [6, 8, 10]


## Task 4: Apply `flatMap()` Transformation

In [10]:
sentences = sc.parallelize([
    "Hello World",
    "Apache Spark is Fast",
    "PySpark is Python plus Spark"
])

In [11]:
# Task A: Split into words (use flatMap)
words = sentences.flatMap(lambda line: line.split(" "))
print(f"Words: {words.collect()}")

Words: ['Hello', 'World', 'Apache', 'Spark', 'is', 'Fast', 'PySpark', 'is', 'Python', 'plus', 'Spark']


In [12]:
# Task B: Create pairs of (word, length)
word_lengths = words.map(lambda w: (w, len(w)))
print(f"Word lengths: {word_lengths.collect()}")

Word lengths: [('Hello', 5), ('World', 5), ('Apache', 6), ('Spark', 5), ('is', 2), ('Fast', 4), ('PySpark', 7), ('is', 2), ('Python', 6), ('plus', 4), ('Spark', 5)]


## Task 5: Chain Transformations

Pipeline: extract only `ERROR` lines, split into words, convert to uppercase.

In [13]:
logs = sc.parallelize([
    "INFO: User logged in",
    "ERROR: Connection failed",
    "INFO: Data processed",
    "ERROR: Timeout occurred",
    "DEBUG: Cache hit"
])

In [14]:
# 1. Filter to keep only ERROR lines
# 2. Split each line into words
# 3. Convert each word to uppercase
error_words = (
    logs
    .filter(lambda line: line.startswith("ERROR"))
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: word.upper())
)
print(f"Error words: {error_words.collect()}")

Error words: ['ERROR:', 'CONNECTION', 'FAILED', 'ERROR:', 'TIMEOUT', 'OCCURRED']


In [15]:
sc.stop()